In [2]:
import pandas as pd
pd.set_option("display.max_columns", 50)

tr = pd.read_csv("../data/ieee/train_transaction.csv")
idn = pd.read_csv("../data/ieee/train_identity.csv")
df = tr.merge(idn, on="TransactionID", how="left")
print(df.shape, "fraud rate:", df["isFraud"].mean())
print("rows with identity info:", df["DeviceType"].notna().mean())

(590540, 434) fraud rate: 0.03499000914417313
rows with identity info: 0.23844278118332374


In [3]:
miss = df.isna().mean().sort_values(ascending=False)
print((miss > 0.5).sum(), "columns are more than 50% empty")
print(miss.head(10))
print(df.dtypes.value_counts())

214 columns are more than 50% empty
id_24    0.991962
id_25    0.991310
id_07    0.991271
id_08    0.991271
id_21    0.991264
id_26    0.991257
id_27    0.991247
id_23    0.991247
id_22    0.991247
dist2    0.936284
dtype: float64
float64    399
str         31
int64        4
Name: count, dtype: int64


In [4]:
df["day"] = df["TransactionDT"] // 86400
print(df["day"].min(), df["day"].max())
print(df.groupby(pd.cut(df["day"], 6))["isFraud"].agg(["count", "mean"]))

1 182
                     count      mean
day                                 
(0.819, 31.167]     137321  0.025852
(31.167, 61.333]     89528  0.039541
(61.333, 91.5]       93085  0.040737
(91.5, 121.667]      97625  0.039324
(121.667, 151.833]   83655  0.033806
(151.833, 182.0]     89326  0.034861


C:\Users\Vale\AppData\Local\Temp\ipykernel_32952\3253634774.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["day"] = df["TransactionDT"] // 86400


In [5]:
for col in ["ProductCD", "card4", "card6", "DeviceType"]:
    print(df.groupby(col, dropna=False)["isFraud"].agg(["count", "mean"]), "\n")

            count      mean
ProductCD                  
C           68519  0.116873
H           33024  0.047662
R           37699  0.037826
S           11628  0.058996
W          439670  0.020399 

                   count      mean
card4                             
american express    8328  0.028698
discover            6651  0.077282
mastercard        189217  0.034331
visa              384767  0.034756
NaN                 1577  0.025999 

                  count      mean
card6                            
charge card          15  0.000000
credit           148986  0.066785
debit            439938  0.024263
debit or credit      30  0.000000
NaN                1571  0.024825 

             count      mean
DeviceType                  
desktop      85165  0.065215
mobile       55645  0.101662
NaN         449730  0.021017 



In [6]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_curve, confusion_matrix

df = df.copy()   # defragment

# float64 -> float32 to halve memory
for c in df.select_dtypes("float64").columns:
    df[c] = df[c].astype("float32")

# every non-numeric column -> integer codes (missing becomes -1)
cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
print(len(cat_cols), "text columns:", cat_cols[:6], "...")
for c in cat_cols:
    df[c] = df[c].astype("category").cat.codes

# time-based split
df = df.sort_values("TransactionDT").reset_index(drop=True)
n = len(df)
train = df.iloc[: int(n * 0.70)]
val   = df.iloc[int(n * 0.70): int(n * 0.85)]
test  = df.iloc[int(n * 0.85):]

drop = ["isFraud", "TransactionID", "TransactionDT", "day"]
feats = [c for c in df.columns if c not in drop]
print(len(feats), "features")
for name, s in [("train", train), ("val", val), ("test", test)]:
    print(name, len(s), s["isFraud"].sum(), round(s["isFraud"].mean(), 4))

31 text columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1'] ...
431 features
train 413378 14538 0.0352
val 88581 3042 0.0343
test 88581 3083 0.0348


In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

def evaluate(y, p, name, thr=0.5):
    fpr, tpr, _ = roc_curve(y, p)
    print(f"{name}: PR-AUC={average_precision_score(y, p):.4f} | recall@1%FPR={tpr[fpr <= 0.01].max():.4f}")
    print(confusion_matrix(y, p >= thr))

base_feats = ["TransactionAmt", "ProductCD", "card4", "card6", "addr1", "P_emaildomain", "DeviceType"]
lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                   LogisticRegression(class_weight="balanced", max_iter=1000))
lr.fit(train[base_feats], train["isFraud"])
evaluate(test["isFraud"], lr.predict_proba(test[base_feats])[:, 1], "LogReg (7 features)")

LogReg (7 features): PR-AUC=0.1091 | recall@1%FPR=0.0720
[[65054 20444]
 [ 1163  1920]]


In [8]:
pos_weight = (train["isFraud"] == 0).sum() / (train["isFraud"] == 1).sum()
model = xgb.XGBClassifier(
    n_estimators=1000, learning_rate=0.05, max_depth=8,
    subsample=0.8, colsample_bytree=0.5, scale_pos_weight=pos_weight,
    tree_method="hist", eval_metric="aucpr", early_stopping_rounds=50, n_jobs=-1,
)
model.fit(train[feats], train["isFraud"], eval_set=[(val[feats], val["isFraud"])], verbose=100)
evaluate(test["isFraud"], model.predict_proba(test[feats])[:, 1], "XGBoost (all features)")

[0]	validation_0-aucpr:0.32165
[100]	validation_0-aucpr:0.51334
[200]	validation_0-aucpr:0.54899
[300]	validation_0-aucpr:0.56049
[400]	validation_0-aucpr:0.56476
[500]	validation_0-aucpr:0.56807
[600]	validation_0-aucpr:0.57027
[700]	validation_0-aucpr:0.57160
[729]	validation_0-aucpr:0.57172
XGBoost (all features): PR-AUC=0.5155 | recall@1%FPR=0.4450
[[82598  2900]
 [ 1394  1689]]


In [9]:
pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False).head(15)

V258    0.104372
V257    0.059768
V218    0.046918
V201    0.035796
V70     0.028828
V294    0.026808
V264    0.012732
V199    0.012635
V91     0.010579
V30     0.009934
V243    0.009852
C4      0.009075
C5      0.008799
V317    0.008292
V187    0.006699
dtype: float32

In [10]:
from sklearn.metrics import precision_recall_curve

pv = model.predict_proba(val[feats])[:, 1]
pt = model.predict_proba(test[feats])[:, 1]
y = test["isFraud"].values

def threshold_for_recall(y_true, p, target):
    prec, rec, thr = precision_recall_curve(y_true, p)
    ok = np.where(rec[:-1] >= target)[0]
    return thr[ok[-1]]          # highest threshold that still reaches the target recall

for target in [0.55, 0.70, 0.80, 0.85]:
    t = threshold_for_recall(val["isFraud"], pv, target)
    pred = pt >= t
    tp = (pred & (y == 1)).sum(); fp = (pred & (y == 0)).sum(); fn = ((~pred) & (y == 1)).sum()
    print(f"target recall {target:.0%}: threshold={t:.3f} | caught={tp} missed={fn} ({fn/y.sum():.1%}) "
          f"| false alarms={fp} | precision={tp/(tp+fp):.1%} | flagged={pred.mean():.1%} of all transactions")

target recall 55%: threshold=0.584 | caught=1560 missed=1523 (49.4%) | false alarms=1896 | precision=45.1% | flagged=3.9% of all transactions
target recall 70%: threshold=0.322 | caught=2029 missed=1054 (34.2%) | false alarms=6668 | precision=23.3% | flagged=9.8% of all transactions
target recall 80%: threshold=0.199 | caught=2316 missed=767 (24.9%) | false alarms=12783 | precision=15.3% | flagged=17.0% of all transactions
target recall 85%: threshold=0.138 | caught=2517 missed=566 (18.4%) | false alarms=18620 | precision=11.9% | flagged=23.9% of all transactions


In [11]:
# df is sorted by TransactionDT; every feature below looks only at earlier rows
g = df.groupby("card1")
df["card1_prev_count"] = g.cumcount()                                   # how many earlier transactions on this card1
df["card1_prev_amt_mean"] = g["TransactionAmt"].transform(lambda s: s.shift().expanding().mean())
df["amt_vs_card_mean"] = df["TransactionAmt"] / df["card1_prev_amt_mean"]
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["n_missing"] = df[feats].isna().sum(axis=1)                          # numeric columns only; text codes use -1

feats2 = feats + ["card1_prev_count", "card1_prev_amt_mean", "amt_vs_card_mean", "hour", "n_missing"]

n = len(df)
train = df.iloc[: int(n * 0.70)]
val   = df.iloc[int(n * 0.70): int(n * 0.85)]
test  = df.iloc[int(n * 0.85):]

model2 = xgb.XGBClassifier(
    n_estimators=1000, learning_rate=0.05, max_depth=8,
    subsample=0.8, colsample_bytree=0.5, scale_pos_weight=pos_weight,
    tree_method="hist", eval_metric="aucpr", early_stopping_rounds=50, n_jobs=-1,
)
model2.fit(train[feats2], train["isFraud"], eval_set=[(val[feats2], val["isFraud"])], verbose=100)
evaluate(test["isFraud"], model2.predict_proba(test[feats2])[:, 1], "XGBoost (all + engineered)")

C:\Users\Vale\AppData\Local\Temp\ipykernel_32952\3168288629.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["card1_prev_count"] = g.cumcount()                                   # how many earlier transactions on this card1
C:\Users\Vale\AppData\Local\Temp\ipykernel_32952\3168288629.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["card1_prev_amt_mean"] = g["TransactionAmt"].transform(lambda s: s.shift().expanding().mean())
C:\Users\Vale\AppData\Local\Temp\ipykernel_32952\3168288629.py:5: PerformanceWarning: DataFram

[0]	validation_0-aucpr:0.34003
[100]	validation_0-aucpr:0.51579
[200]	validation_0-aucpr:0.54879
[300]	validation_0-aucpr:0.55952
[400]	validation_0-aucpr:0.56716
[500]	validation_0-aucpr:0.57140
[600]	validation_0-aucpr:0.57543
[700]	validation_0-aucpr:0.57777
[745]	validation_0-aucpr:0.57750
XGBoost (all + engineered): PR-AUC=0.5171 | recall@1%FPR=0.4379
[[83181  2317]
 [ 1438  1645]]


In [12]:
user_feats = ["TransactionAmt", "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
              "addr1", "addr2", "dist1", "P_emaildomain", "R_emaildomain", "DeviceType"]

def add_form_features(d):
    d = d.copy()
    d["amt_cents"] = (d["TransactionAmt"] * 100).round() % 100
    d["email_match"] = ((d["P_emaildomain"] == d["R_emaildomain"]) & (d["P_emaildomain"] != -1)).astype(int)
    return d

tr2, va2, te2 = add_form_features(train), add_form_features(val), add_form_features(test)
feats_form = user_feats + ["amt_cents", "email_match"]

model_form = xgb.XGBClassifier(
    n_estimators=1000, learning_rate=0.05, max_depth=7,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
    tree_method="hist", eval_metric="aucpr", early_stopping_rounds=50, n_jobs=-1,
)
model_form.fit(tr2[feats_form], tr2["isFraud"], eval_set=[(va2[feats_form], va2["isFraud"])], verbose=100)
evaluate(te2["isFraud"], model_form.predict_proba(te2[feats_form])[:, 1], "XGBoost (form features)")
pd.Series(model_form.feature_importances_, index=feats_form).sort_values(ascending=False)

[0]	validation_0-aucpr:0.16193
[61]	validation_0-aucpr:0.19483
XGBoost (form features): PR-AUC=0.1906 | recall@1%FPR=0.1482
[[70874 14624]
 [ 1183  1900]]


email_match       0.221199
R_emaildomain     0.186675
amt_cents         0.108516
card6             0.100053
ProductCD         0.065020
addr2             0.057802
card3             0.049997
DeviceType        0.035723
TransactionAmt    0.032568
card2             0.023563
P_emaildomain     0.022747
addr1             0.021497
card5             0.021352
card1             0.019985
dist1             0.018931
card4             0.014372
dtype: float32

In [13]:
pv_f = model_form.predict_proba(va2[feats_form])[:, 1]
pt_f = model_form.predict_proba(te2[feats_form])[:, 1]
y = te2["isFraud"].values

for target in [0.30, 0.40, 0.50, 0.60]:
    t = threshold_for_recall(va2["isFraud"], pv_f, target)
    pred = pt_f >= t
    tp = (pred & (y == 1)).sum(); fp = (pred & (y == 0)).sum(); fn = ((~pred) & (y == 1)).sum()
    print(f"target {target:.0%}: thr={t:.3f} | caught={tp} missed={fn} | false alarms={fp} "
          f"| precision={tp/(tp+fp):.1%} | flagged={pred.mean():.1%} | cost={fn*100 + fp*5}")

target 30%: thr=0.633 | caught=899 missed=2184 | false alarms=2491 | precision=26.5% | flagged=3.8% | cost=230855
target 40%: thr=0.573 | caught=1248 missed=1835 | false alarms=5064 | precision=19.8% | flagged=7.1% | cost=208820
target 50%: thr=0.541 | caught=1532 missed=1551 | false alarms=8522 | precision=15.2% | flagged=11.4% | cost=197710
target 60%: thr=0.498 | caught=1917 missed=1166 | false alarms=15084 | precision=11.3% | flagged=19.2% | cost=192020


In [14]:
for depth, mcw in [(4, 50), (5, 100), (6, 200)]:
    m = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.03, max_depth=depth, min_child_weight=mcw,
        subsample=0.8, colsample_bytree=0.6, reg_lambda=10, scale_pos_weight=pos_weight,
        tree_method="hist", eval_metric="aucpr", early_stopping_rounds=50, n_jobs=-1,
    )
    m.fit(tr2[feats_form], tr2["isFraud"], eval_set=[(va2[feats_form], va2["isFraud"])], verbose=0)
    print(depth, mcw, "best val PR-AUC:", round(m.best_score, 4), "rounds:", m.best_iteration)

4 50 best val PR-AUC: 0.174 rounds: 19
5 100 best val PR-AUC: 0.2179 rounds: 996
6 200 best val PR-AUC: 0.1857 rounds: 14


In [15]:
model_form2 = xgb.XGBClassifier(
    n_estimators=3000, learning_rate=0.03, max_depth=5, min_child_weight=100,
    subsample=0.8, colsample_bytree=0.6, reg_lambda=10, scale_pos_weight=pos_weight,
    tree_method="hist", eval_metric="aucpr", early_stopping_rounds=100, n_jobs=-1,
)
model_form2.fit(tr2[feats_form], tr2["isFraud"], eval_set=[(va2[feats_form], va2["isFraud"])], verbose=250)
print("best val PR-AUC:", round(model_form2.best_score, 4), "| best round:", model_form2.best_iteration)

evaluate(te2["isFraud"], model_form2.predict_proba(te2[feats_form])[:, 1], "XGBoost (form, regularised)")

pv_f = model_form2.predict_proba(va2[feats_form])[:, 1]
pt_f = model_form2.predict_proba(te2[feats_form])[:, 1]
y = te2["isFraud"].values
for thr in [0.4, 0.5, 0.6]:
    pred = pt_f >= thr
    tp = (pred & (y == 1)).sum(); fp = (pred & (y == 0)).sum(); fn = ((~pred) & (y == 1)).sum()
    print(f"thr={thr}: caught={tp} missed={fn} false alarms={fp} precision={tp/(tp+fp):.1%} "
          f"flagged={pred.mean():.1%} cost={fn*100 + fp*5}")

[0]	validation_0-aucpr:0.13992
[250]	validation_0-aucpr:0.19534
[500]	validation_0-aucpr:0.20624
[750]	validation_0-aucpr:0.21256
[1000]	validation_0-aucpr:0.21784
[1250]	validation_0-aucpr:0.22323
[1500]	validation_0-aucpr:0.22725
[1750]	validation_0-aucpr:0.22962
[2000]	validation_0-aucpr:0.23330
[2250]	validation_0-aucpr:0.23691
[2500]	validation_0-aucpr:0.23910
[2750]	validation_0-aucpr:0.24121
[2999]	validation_0-aucpr:0.24404
best val PR-AUC: 0.244 | best round: 2999
XGBoost (form, regularised): PR-AUC=0.2187 | recall@1%FPR=0.1576
[[74209 11289]
 [ 1189  1894]]
thr=0.4: caught=2154 missed=929 false alarms=17106 precision=11.2% flagged=21.7% cost=178430
thr=0.5: caught=1894 missed=1189 false alarms=11289 precision=14.4% flagged=14.9% cost=175345
thr=0.6: caught=1629 missed=1454 false alarms=7297 precision=18.3% flagged=10.1% cost=181885


In [16]:
import json

cat_form_cols = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain", "DeviceType"]

raw_tr = pd.read_csv("../data/ieee/train_transaction.csv",
                     usecols=["TransactionID", "ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"])
raw_id = pd.read_csv("../data/ieee/train_identity.csv", usecols=["TransactionID", "DeviceType"])
raw = raw_tr.merge(raw_id, on="TransactionID", how="left")

mappings = {}
for c in cat_form_cols:
    cats = sorted(raw[c].dropna().unique().tolist())
    mappings[c] = {v: i for i, v in enumerate(cats)}
    chk = raw[["TransactionID", c]].merge(df[["TransactionID", c]], on="TransactionID", suffixes=("_raw", "_code"))
    expected = chk[c + "_raw"].map(mappings[c]).fillna(-1).astype(int)
    assert (expected.values == chk[c + "_code"].values).all(), f"mapping mismatch in {c}"
    print(c, len(cats), "categories, mapping verified")

model_form2.save_model("../models/ecommerce_xgb.ubj")
meta = {"features": feats_form, "threshold": 0.5, "categorical": mappings,
        "required": ["TransactionAmt", "ProductCD"]}
json.dump(meta, open("../models/ecommerce_meta.json", "w"), indent=1)

ProductCD 5 categories, mapping verified
card4 4 categories, mapping verified
card6 4 categories, mapping verified
P_emaildomain 59 categories, mapping verified
R_emaildomain 60 categories, mapping verified
DeviceType 2 categories, mapping verified
